## 01 EDA (복약)

식약처 의약품 7종 데이터의 분포·결측·중복·인코딩을 탐색하고 후속 적재 전제를 점검합니다.

## 환경 설정 및 라이브러리 임포트

In [ ]:
"""01_eda — 복약 가이드 영역 EDA.

식약처 공공데이터 7종 CSV/XLSX의 구조를 탐색하고 행수를 검증한다.
ERD 매핑 검토 및 마스터 테이블 분리 여부 판단을 위한 기초 자료를 생성한다.
"""

# 표준 라이브러리
import csv as csv_module
import os

# 서드파티 라이브러리
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

## 환경 감지 및 경로 설정

Google Colab과 로컬 PyCharm 환경 모두에서 동작하도록 자동 감지합니다.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AH_03_06_medication_data'
    ENV = 'Colab'
except ImportError:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../data/raw/medication'))
    ENV = 'Local'

print(f'환경: {ENV}')
print(f'BASE_DIR: {BASE_DIR}')

assert os.path.isdir(BASE_DIR), (
    f'데이터 폴더가 없습니다: {BASE_DIR}\n'
    f'셋업 가이드: AH_03_06_medication_data 폴더를 다음 위치에 두세요.\n'
    f'  - Colab: MyDrive 루트\n'
    f'  - 로컬: AH_03_06/ml/data/raw/medication/'
)
print('데이터 폴더 확인됨')

## 데이터 파일 메타 정보 (7종)

식약처 11종 중 ERD 변경 제안서(2026-05-14)에 반영된 7종만 사용합니다.
데이터분석_보고서.md 기준 행수를 expected_rows로 보유합니다.

In [ ]:
CSV_FILES = {
    '1_item_permit': {
        'filename': 'OpenData_ItemPermitC20260513.csv',
        'korean_name': '의약품 제품허가목록',
        'expected_rows': 43276,
        'erd_table': 'drug_info',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': False,
    },
    '2_item_permit_detail': {
        'filename': 'OpenData_ItemPermitDetail20260513.xls',
        'korean_name': '의약품 제품허가 상세정보',
        'expected_rows': 43276,
        'erd_table': 'drug_info_detail',
        'encoding': None,
        'is_xlsx_disguised': True,
        'quoting_issue': False,
    },
    '5_easy_excel': {
        'filename': 'OpenData_EasyExcelListC20260513.csv',
        'korean_name': 'e약은요정보',
        'expected_rows': 4762,
        'erd_table': 'drug_info_detail',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': False,
    },
    '6_day_max_dosg': {
        'filename': 'OpenData_DayMaxDosgQyInfoC20260513.csv',
        'korean_name': '1일최대투여량',
        'expected_rows': 4962,
        'erd_table': 'drug_dose_limit',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': False,
    },
    '7_recall': {
        'filename': 'OpenData_PotOpenRecallSaleStopC20260513.csv',
        'korean_name': '회수판매중지',
        'expected_rows': 933,
        'erd_table': 'drug_info (is_recalled/recall_* 컬럼)',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': False,
    },
    '10_dur_ingr': {
        'filename': 'OpenData_PotOpenDurIngr_AC20260513.csv',
        'korean_name': 'DUR유형별 성분 현황',
        'expected_rows': 1742,
        'erd_table': 'dur_concurrent_ingredient',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': True,
    },
    '11_dur_item': {
        'filename': 'OpenData_PotOpenDurItem_AC20260513.csv',
        'korean_name': 'DUR유형별 품목 현황',
        'expected_rows': 404121,
        'erd_table': 'dur_concurrent_product',
        'encoding': 'utf-8-sig',
        'is_xlsx_disguised': False,
        'quoting_issue': False,
        'large_file': True,
    },
}

print(f'총 {len(CSV_FILES)}개 파일 메타 정보 등록됨')
for key, meta in CSV_FILES.items():
    print(f"  [{key}] {meta['korean_name']} (기대 {meta['expected_rows']:,}행) → {meta['erd_table']}")

## 데이터 로딩 함수 정의

데이터분석_보고서.md의 특이사항을 함수 내부에서 처리합니다.
- xlsx로 위장한 .xls (#2): openpyxl 엔진
- 따옴표 짝 이슈 (#10): QUOTE_NONE 폴백

In [ ]:
def load_drug_data(meta: dict, base_dir: str) -> pd.DataFrame:
    """식약처 의약품 데이터 로딩 (xlsx 위장 / quoting 이슈 처리)"""
    path = os.path.join(base_dir, meta['filename'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'파일 없음: {path}')

    # xlsx 위장한 .xls (#2)
    if meta.get('is_xlsx_disguised'):
        return pd.read_excel(path, engine='openpyxl')

    # 일반 CSV 로딩
    try:
        return pd.read_csv(path, encoding=meta['encoding'], low_memory=False)
    except Exception as e:
        # quoting 이슈 폴백 (#10 DUR 성분)
        if meta.get('quoting_issue'):
            print(f'  {meta["filename"]} quoting 이슈 → QUOTE_NONE 폴백')
            return pd.read_csv(
                path,
                encoding=meta['encoding'],
                quoting=csv_module.QUOTE_NONE,
                low_memory=False,
            )
        raise

print('load_drug_data 함수 정의 완료')

## 7종 CSV 일괄 로딩 및 행수 검증

데이터분석_보고서.md의 expected_rows와 실제 로드된 행수를 비교합니다.
불일치 시 AssertionError 발생.

In [ ]:
dfs = {}

for key, meta in CSV_FILES.items():
    print(f'\n로딩 중: [{key}] {meta["korean_name"]} ...')
    df = load_drug_data(meta, BASE_DIR)
    dfs[key] = df

    actual = len(df)
    expected = meta['expected_rows']
    status = '[OK]' if actual == expected else '[WARN]'
    print(f'  {status} 행수: {actual:,} (기대 {expected:,})')

    # quoting 이슈가 있는 파일(#10 DUR 성분)은 QUOTE_NONE 폴백 시 일부 행 손실 가능
    # 데이터분석_보고서.md 기준 약 ±100건까지 허용
    if meta.get('quoting_issue'):
        diff = abs(actual - expected)
        assert diff <= 100, (
            f'행수 차이 큼 [{key}]: 실제 {actual:,}행 / 기대 {expected:,}행 (차이 {diff:,})'
        )
    elif meta.get('large_file'):
        # 대용량 CSV는 임베디드 개행 등으로 파서 간 미세 차이 발생 가능
        diff = abs(actual - expected)
        assert diff <= 500, (
            f'행수 차이 큼 [{key}]: 실제 {actual:,}행 / 기대 {expected:,}행 (차이 {diff:,})'
        )
    else:
        assert actual == expected, (
            f'행수 불일치 [{key}]: 실제 {actual:,}행 / 기대 {expected:,}행'
        )

print(f'\n7종 모두 로딩 및 행수 검증 완료')
print(f'총 행수: {sum(len(df) for df in dfs.values()):,}')

## 각 CSV별 기본 정보

shape, 메모리 사용량, 결측 비율(10% 이상), 첫 5행을 출력합니다.

In [ ]:
for key, df in dfs.items():
    meta = CSV_FILES[key]
    print('\n' + '=' * 80)
    print(f"[{key}] {meta['korean_name']} → ERD: {meta['erd_table']}")
    print('=' * 80)
    print(f'shape: {df.shape}')
    print(f'메모리: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

    null_ratio = df.isnull().mean()
    high_null = null_ratio[null_ratio > 0.1].sort_values(ascending=False)
    if len(high_null) > 0:
        print(f'\n결측 비율 10% 이상 컬럼:')
        for col, ratio in high_null.items():
            print(f'  {col}: {ratio*100:.1f}%')
    else:
        print('\n결측 비율 10% 이상 컬럼: 없음')

    print(f'\n첫 3행 (일부 컬럼):')
    preview_cols = df.columns[:5].tolist()
    print(df[preview_cols].head(3).to_string())

## 핵심 컬럼 unique 값 (향후 마스터 분리 검토용)

ERD 변경 제안서 정규화 정책 메모에 따라, route·dur_type·dosage_form 등의 코드성 컬럼은
VARCHAR로 직접 저장합니다. 다만 향후 마스터 테이블로 분리 시 시드 데이터로 활용하기 위해
현재 unique 값을 미리 확보합니다.

In [ ]:
ENUM_COLUMNS = {
    '1_item_permit': ['전문일반구분', '신고허가구분', '분류명'],
    '6_day_max_dosg': ['제형명', '투여경로', '투여단위'],
    '7_recall': ['강제여부구분'],
    '10_dur_ingr': ['DUR유형', '단일복합구분코드', '등급'],
    '11_dur_item': ['DUR유형명', '제형구분명', '전문일반구분명'],
}

for key, cols in ENUM_COLUMNS.items():
    df = dfs[key]
    meta = CSV_FILES[key]
    print('\n' + '=' * 80)
    print(f"[{key}] {meta['korean_name']}")
    print('=' * 80)
    for col in cols:
        if col not in df.columns:
            print(f'  컬럼 없음: {col}')
            continue
        uniq = df[col].dropna().unique()
        sample = list(uniq)[:10]
        suffix = ' ...' if len(uniq) > 10 else ''
        print(f'\n  {col} (총 {len(uniq)}개): {sample}{suffix}')

In [ ]:
# 7종 데이터프레임을 processed/medication/{key}.pkl로 캐시
# (02·03 노트북에서 read_pickle로 빠른 로드)
import os
PROCESSED_DIR = os.path.abspath(os.path.join(BASE_DIR, '../../processed/medication'))
os.makedirs(PROCESSED_DIR, exist_ok=True)

for key, df in dfs.items():
    out_path = os.path.join(PROCESSED_DIR, f'{key}.pkl')
    df.to_pickle(out_path)
    print(f'저장: {key} → {out_path}')

## 작업 마무리 및 다음 단계

### 본 노트북에서 확인한 사항

- 7종 CSV 모두 정상 로딩 (총 약 500,000행)
- 데이터분석_보고서.md 기준 행수와 일치 (파서 특이사항이 있는 두 파일은 사전 합의된 허용 오차 범위 내: #10 DUR 성분 ±100건, #11 DUR 품목 ±500건)
- 인코딩(UTF-8-SIG), xlsx 위장(.xls), quoting 이슈 함수 내부에서 처리
- 핵심 코드성 컬럼의 unique 값 확인 (향후 마스터 분리 시 시드 활용 가능)
- 7종 데이터프레임을 `processed/medication/{key}.pkl`로 캐시 저장 (02·03 노트북 빠른 로딩용)

### 다음 노트북 (02_drug_matching)

- 사용자 입력 약품명을 `1_item_permit` · `2_item_permit_detail` · `5_easy_excel`와 매칭
- 정확 매칭 (품목일련번호) + 모호 매칭 (rapidfuzz WRatio) + 자모 분해 fuzzy + ATC tiebreak
- ERD: drug_info + drug_info_detail 적재 검증

### 다음 노트북 (03_dur_safety_check)

- DUR 병용금기 2단 매칭: 제품(`11_dur_item`) → 성분(`10_dur_ingr`) 폴백
- 1일 최대량(`6_day_max_dosg`) 초과 검증
- 회수약 검출(`7_recall`)
- ERD: drug_dose_limit + dur_concurrent_* + drug_ingredient_map